In [11]:
from vllm import LLM, SamplingParams

In [12]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [20]:
# llm = LLM(
#     model="meta-llama/Llama-3.2-3B-Instruct",
#     dtype="float16",  # if you're using fp16 to save memory
#     max_model_len=2048,  
#     gpu_memory_utilization=0.85 
# )

In [21]:
llm = LLM(
    model="./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1",
    dtype="float16",  # if you're using fp16 to save memory
    max_model_len=2048,  
    gpu_memory_utilization=0.85 
)

WARNING 04-06 13:50:21 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 04-06 13:50:21 [config.py:585] This model supports multiple tasks: {'classify', 'generate', 'score', 'reward', 'embed'}. Defaulting to 'generate'.
INFO 04-06 13:50:21 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-06 13:50:21 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', speculative_config=None, tokenizer='./lora/lora_16bit_merged_3b_r64_s1000_i1000_v1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_bac

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-06 13:50:25 [loader.py:447] Loading weights took 2.32 seconds
INFO 04-06 13:50:25 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.475678 seconds
INFO 04-06 13:50:31 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/2d3e90b0c6/rank_0_0 for vLLM's torch.compile
INFO 04-06 13:50:31 [backends.py:425] Dynamo bytecode transform time: 5.95 s


[rank0]:W0406 13:50:32.202000 63031 site-packages/torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 04-06 13:50:33 [backends.py:132] Cache the graph of shape None for later use
INFO 04-06 13:50:49 [backends.py:144] Compiling a graph for general shape takes 17.80 s
INFO 04-06 13:50:59 [monitor.py:33] torch.compile takes 23.75 s in total
INFO 04-06 13:51:00 [kv_cache_utils.py:566] GPU KV cache size: 19,168 tokens
INFO 04-06 13:51:00 [kv_cache_utils.py:569] Maximum concurrency for 2,048 tokens per request: 9.36x
INFO 04-06 13:51:16 [gpu_model_runner.py:1534] Graph capturing finished in 16 secs, took 0.43 GiB
INFO 04-06 13:51:16 [core.py:151] init engine (profile, create kv cache, warmup model) took 50.88 seconds


In [22]:
#alpaca prompt
instruction = """Generate SVG code to visually represent the following text description, while respecting the given constraints.
        <constraints>
        * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
        * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
        </constraints>
        
        <example>
            <description>"A red circle with a blue square inside"</description>
            
            ```svg
            <svg viewBox="0 0 256 256" width="256" height="256">
              <circle cx="50" cy="50" r="40" fill="red"/>
              <rect x="30" y="30" width="40" height="40" fill="blue"/>
              <...>
               ...
              <...>
            </svg>
        ```
        </example>        
        
        Please ensure that the generated SVG code is well-formed, valid, and strictly adheres to these constraints.
        Focus on a clear and concise representation of the input description within the given limitations. 
        Always give the complete SVG code with nothing omitted. Never use an ellipsis.
        Do not include unnecessary explanations. Just give the code.
        """
        
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        {}
    
        ### Input:             
        <description>"{}"</description>
    
        ### Response:
        """
formatted_input = alpaca_prompt.format(instruction, 'Sun rising in the East')

In [23]:
formatted_input

'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n    \n        ### Instruction:\n        Generate SVG code to visually represent the following text description, while respecting the given constraints.\n        <constraints>\n        * **Allowed Elements:** `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`\n        * **Allowed Attributes:** `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`\n        </constraints>\n        \n        <example>\n            <description>"A red circle with a blue square inside"</description>\n            \n            ```svg\n            <svg viewBox="0 0 256 256" width="256" height="256">\n              <circle cx="50" cy="50" r="40" fill="red"/>\n       

In [24]:
sampling_params = SamplingParams(temperature=0.5, top_p=0.95,max_tokens=1024)
outputs = llm.generate([formatted_input], sampling_params)

for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(generated_text)

Processed prompts: 100%|█| 1/1 [00:03<00:00,  3.42s/it, est. speed input: 112.67

 <svg viewBox="0 0 256 256" width="256" height="256" xmlns="http://www.w3.org/2000/svg">
          <defs>
            <linearGradient id="sunGradient" x1="0" y1="0" x2="0" y2="1">
              <stop offset="0%" stop-color="yellow" />
              <stop offset="100%" stop-color="orange" />
            </linearGradient>
            <radialGradient id="skyGradient" cx="0.5" cy="0.5" r0="0" r1="1">
              <stop offset="0%" stop-color="blue" />
              <stop offset="100%" stop-color="darkBlue" />
            </radialGradient>
          </defs>
          <rect x="0" y="0" width="256" height="256" fill="url(#skyGradient)" />
          <circle cx="128" cy="128" r="30" fill="url(#sunGradient)" />
          <polyline points="150,50 160,100 130,100" fill="none" stroke="orange" stroke-width="2" />
          <line x1="0" y1="100" x2="180" y2="100" stroke="orange" stroke-width="2" />
        </svg>
